# Devoir #1 (À rendre le 09/02/2026, 23h59)

**ELEC70122: ML pour la Prise de Décision Critique en Sécurité**<br>
**Instructrice: Sonali Parbhoo**<br>
**Automne 2026**

**Nom:**

### Instructions:

**Format de soumission:** Utilisez ce notebook comme modèle pour compléter votre devoir. Veuillez intercaler des blocs de texte (en utilisant des cellules Markdown) parmi le code `python` et les résultats -- formatez votre soumission pour une lisibilité maximale. Vos devoirs seront notés sur la correction ainsi que sur la clarté de l'exposition et de la présentation -- une réponse "correcte" seule sans explication ou présentée dans un format difficile à suivre ne recevra aucun crédit.

**Vérification du code:** Avant de soumettre, vous devez faire un "Redémarrer et Tout Exécuter" sous "Kernel" dans le menu Jupyter ou Colab. ***Les parties de votre soumission qui contiennent des erreurs syntaxiques ou d'exécution ne seront pas notées***.

**Bibliothèques et packages:** Sauf si un problème vous demande spécifiquement d'implémenter à partir de zéro, vous pouvez utiliser n'importe quel package de bibliothèque `python` dans la distribution standard Anaconda.

In [2]:
### Importer les bibliothèques de base
import numpy as np
import pandas as pd
import sklearn as sk
from scipy.stats import multivariate_normal
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.datasets import make_classification, make_regression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import log_loss
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
def get_posterior_samples(prior_var, noise_var, x_matrix, y_matrix, x_test_matrix, samples=100):
    '''Fonction pour générer des échantillons prédictifs postérieurs pour un modèle de régression linéaire bayésienne'''
    prior_variance = np.diag(prior_var * np.ones(x_matrix.shape[1]))
    prior_precision = np.linalg.inv(prior_variance)

    joint_precision = prior_precision + x_matrix.T.dot(x_matrix) / noise_var
    joint_variance = np.linalg.inv(joint_precision)
    joint_mean = joint_variance.dot(x_matrix.T.dot(y_matrix)) / noise_var

    # échantillonnage de 100 points du postérieur
    posterior_samples = np.random.multivariate_normal(joint_mean.flatten(), joint_variance, size=samples)

    # prendre des échantillons prédictifs postérieurs
    posterior_predictions = np.dot(posterior_samples, x_test_matrix.T)
    posterior_predictive_samples = posterior_predictions[np.newaxis, :, :] + np.random.normal(0, noise_var**0.5, size=(100, posterior_predictions.shape[0], posterior_predictions.shape[1]))
    posterior_predictive_samples = posterior_predictive_samples.reshape((100 * posterior_predictions.shape[0], posterior_predictions.shape[1]))
    return posterior_predictions, posterior_predictive_samples


def generate_data(number_of_points=10, noise_variance=0.3):
    '''Fonction pour générer des données de régression jouet'''
    # x d'entraînement
    x_train = np.hstack((np.linspace(-1, -0.5, number_of_points), np.linspace(0.5, 1, number_of_points)))
    # fonction reliant x et y
    f = lambda x: 3 * x**3
    # y est égal à f(x) plus du bruit gaussien
    y_train = f(x_train) + np.random.normal(0, noise_variance**0.5, 2 * number_of_points)
    x_test = np.array(list(set(list(np.hstack((np.linspace(-1, 1, 200), x_train))))))
    x_test = np.sort(x_test)
    return x_train, y_train, x_test

: 

## Partie I: Estimateurs du Maximum de Vraisemblance pour la Régression Polynomiale

Dans ce problème, on vous donne une fonction, `generate_data`, pour générer des jeux de données jouets avec un seul prédicteur $X$ représentant l'âge du patient (normalisé) et une seule variable de sortie $y$ représentant la pression artérielle diastolique (normalisée et redimensionnée), et votre tâche est d'ajuster des modèles polynomiaux aux données. C'est-à-dire, supposez que la sortie $y$ peut être modélisée par le processus suivant:

\begin{align}
y &= f(x) + \epsilon = w_0 + w_1x + w_2x^2 + \ldots + w_Dx^D + \epsilon, \quad \epsilon \sim \mathcal{N}(0, 0.3)
\end{align}

où les $w_d$, les *paramètres* de la fonction $f$, sont des constantes inconnues, et le degré $D$ est un hyperparamètre.


Vous remarquerez que dans ces jeux de données, l'entrée de test est échantillonnée à partir d'une distribution différente de l'entrée d'entraînement: l'entrée d'entraînement a un écart, il n'y a pas de valeurs d'entrée d'entraînement dans [-0.5, 0.5], alors que l'entrée de test est échantillonnée sur [-1, 1]. Ce changement des distributions sur les valeurs de $x$ entre l'entraînement et le test est appelé **décalage de covariables** (covariate shift).

Ces jeux de données jouets simulent un problème très courant en apprentissage automatique: les modèles sont ajustés sur des données d'entraînement, mais pendant le déploiement, ils reçoivent des données dissemblables aux données d'entraînement (c'est-à-dire que le modèle rencontre un décalage de covariables). En tant que tel, vous devez traiter `x_train`, `y_train` comme des données disponibles pendant le développement et l'évaluation du modèle, et `x_test` comme des données que vous rencontrez pendant le déploiement du modèle.

L'objectif de ce devoir est d'explorer comment gérer le risque d'un modèle déployé sous décalage de covariables. Les idées développées dans ce devoir deviendront un axe majeur dans la dernière partie du cours et le fondement d'un domaine de recherche actif.

1. **(Effet de la Complexité du Modèle)** Générez un jeu de données jouet avec 40 observations (définissez le paramètre `number_of_points=20` pour `generate_data`, puisque deux fois le nombre de `number_of_points` sera généré), puis visualisez l'ajustement des modèles polynomiaux MLE, avec des degrés $D = [1, 3, 5, 10, 15, 20, 50, 100]$ - vous devez entraîner sur `x_train` et **visualiser en prédisant sur `x_test` fourni par la fonction de génération de données (`x_test` est un ensemble plus grand de points de test qui inclut `x_train`)**. Vous devrez réfléchir à votre visualisation pour que ces différents modèles puissent être comparés visuellement de manière significative. <br><br>
Discutez de l'effet du choix du degré polynomial sur l'ajustement du modèle (décrivez concrètement pourquoi certains choix sont non idéaux dans le contexte du problème).

2. **(Sélection de Modèle)** Plus tard dans le cours, nous étudierons un certain nombre de métriques couramment utilisées pour sélectionner entre différents modèles MLE. Toutes ces métriques encodent essentiellement le Rasoir d'Occam: sélectionner la complexité minimale du modèle qui satisfait un objectif de modélisation prédéterminé. <br><br>
Pour l'instant, une méthode très simple pour sélectionner le degré optimal est via la validation croisée (par bootstrap):

  1. échantillonnez aléatoirement deux jeux de données, `x_train`, `x_valid`, à partir de la fonction de génération de données: un pour l'entraînement et un pour la validation. Ajustez un modèle polynomial MLE de degré $d$ sur les données d'entraînement et évaluez ses performances sur les données de validation. Sur $S$ nombre de telles paires de jeux de données échantillonnées aléatoirement, moyennez les performances de validation du modèle.
  2. tracez le score de validation en fonction de la complexité du modèle, le degré polynomial $d$.
  3. basé sur le graphique, sélectionnez le degré minimal qui atteint une haute performance de validation moyenne (c'est-à-dire cherchez le 'coude' du graphique).

  Expliquez pourquoi effectuer la sélection de modèle par validation croisée atténue le risque de choisir un polynôme indésirable (identifié dans le Problème 1)?<br><br>
  Implémentez la sélection de modèle par validation croisée pour le jeu de données jouet généré dans le Problème 1 en utilisant le MSE comme métrique de performance et sélectionnez un degré optimal parmi $D=[1,3,5,10,15,20,50,100]$.

3. **(Estimation de l'Incertitude)** Nous utilisons souvent l'incertitude prédictive bootstrap des modèles MLE comme indicateur de notre confiance dans la sortie du modèle. De plus en plus, en pratique, la prise de décision est déférée aux experts humains lorsque l'incertitude prédictive du modèle est trop élevée. <br><br>
Étant donné votre compréhension du jeu de données (`x_train` et `x_test`), décrivez à quoi l'incertitude du modèle ***devrait*** idéalement ressembler à travers l'espace d'entrée (c'est-à-dire si vous traciez l'incertitude du modèle en fonction de $x$, à quoi ressemblerait-elle)? Justifiez votre réponse: considérez le contexte du problème - l'entrée de test a subi un décalage de covariables et est dissemblable à l'entrée d'entraînement, quel type d'incertitude vous aiderait à atténuer le risque dans cette condition?<br><br>
Une pratique courante pour estimer l'incertitude prédictive est d'ajuster un grand nombre de modèles (bootstrap) sur les données d'entraînement (cette collection de modèles est appelée un **ensemble**), puis, à une entrée $x$, utiliser la variance des prédictions de l'ensemble pour estimer l'incertitude en $x$. Tracez l'intervalle prédictif à 95% de 200 modèles polynomiaux MLE bootstrap pour chaque degré $D=[1,3,5,10,15,20,50,100]$, arrangez vos graphiques comme sous-graphiques dans une seule figure. Pour quel degré polynomial obtenez-vous l'incertitude prédictive la plus idéale (selon votre description ci-dessus)? Est-ce le degré que vous avez sélectionné dans le Problème 2? Expliquez pourquoi vous vous attendriez ou non à ce que le degré optimal dans le Problème 2 produise l'estimation d'incertitude la plus idéale.<br><br>
Faites les mêmes graphiques des intervalles prédictifs à 95% pour les degrés $D=[1,3,5,10,15,20,50,100]$, avec des modèles ajustés sur des jeux de données d'entraînement plus grands - définissez `number_of_points` à 50, 100, 500, 1000 (arrangez tous ces graphiques dans une seule figure). Que se passe-t-il avec les prédictions de l'ensemble dans la région riche en données d'entraînement? Que se passe-t-il avec les prédictions de l'ensemble dans la région pauvre en données d'entraînement? Est-ce des comportements attendus (reliez ce que vous voyez dans les deux cas aux propriétés asymptotiques du MLE)?
<br><br>
Quand les données d'entraînement sont abondantes (`number_of_points=1000`), est-ce que l'un des intervalles prédictifs à 95% est idéal (selon votre description ci-dessus)? Qu'est-ce que cela implique sur la faisabilité d'utiliser la variance des prédictions de l'ensemble pour estimer l'incertitude prédictive à une entrée $x$?

4. **(Effet de la Régularisation)** En pratique, les modèles MLE sont presque toujours entraînés avec régularisation (puisqu'ils ont tendance à surajuster les données d'entraînement). Ici, nous allons explorer l'effet d'ajouter une régularisation $\ell_2$ à nos modèles polynomiaux MLE (c'est-à-dire, utiliser le modèle de régression `Ridge` de `sklearn` après avoir augmenté votre entrée avec des caractéristiques polynomiales). <br><br>
Pour un jeu de données jouet avec 40 observations (`number_of_points=20`), tracez les intervalles prédictifs à 95% pour les degrés $D = [1,3,5,10,15,20,50,100]$ et les forces de régularisation `alpha = [5e-3, 1e-2, 1e-1, 1e0, 1e1]` (vous devez organiser ces graphiques dans une grille).<br><br>
Décrivez l'effet de la régularisation sur les incertitudes bootstrap. En regardant ces résultats, les objectifs de la régularisation $\ell_2$ et l'obtention d'une estimation d'incertitude prédictive utile sont-ils nécessairement bien alignés?

## Partie II: Régression Polynomiale Bayésienne
Dans ce problème, votre tâche est d'effectuer une régression polynomiale bayésienne sur les jeux de données jouets de la Partie I. C'est-à-dire, supposez que la sortie $y$ peut être modélisée par le processus suivant:

\begin{align}
y &= f(x) + \epsilon = w_0 + w_1x + w_2x^2 + \ldots + w_Dx^D + \epsilon, \quad \epsilon \sim N(0, 0.3)\\
w_d &\sim N(0, \alpha)
\end{align}

où $\alpha$ est un hyperparamètre et doit être fixé avant que la modélisation et l'inférence ne commencent.

1. **(Régression à Noyau Bayésienne)** Tout comme nous pouvons traiter un modèle de régression polynomiale comme un modèle de régression multi-linéaire après avoir ***transformé*** les données d'entrée en ajoutant des caractéristiques polynomiales. Nous pouvons traiter la régression polynomiale bayésienne comme une régression linéaire bayésienne sur les entrées transformées. Formellement, l'application qui prend une entrée $\mathbf{x}_n \in \mathbb{R}^{D'}$ et la transforme en une nouvelle entrée $\phi(\mathbf{x}_n) \in \mathbb{R}^{D}$ est appelée une **application de caractéristiques** (feature map), $\phi: \mathbb{R}^{D'} \to \mathbb{R}^{D}$, pour une entrée 1-dimensionnelle $x \in \mathbb{R}$, l'application de caractéristiques polynomiales de degré $D$ est définie par
\begin{align}
\\\phi: \mathbb{R} &\to \mathbb{R}^D\\
x &\mapsto [1, x, x^2, \ldots, x^D]\\
\end{align}
<br> Ainsi, nous pouvons réécrire la régression polynomiale bayésienne comme
\begin{align}
\\y &= \mathbf{w}^\top \mathbf{x} + \epsilon, \quad \epsilon \sim {N}(0, 0.3)\\
\mathbf{w} &\sim {N}(0, \alpha I_{D\times D})\\
\end{align}
<br>Dénotez la matrice $N\times D$ des entrées transformées par $\Phi$, où la $n$-ième ligne de la matrice est la $n$-ième entrée $\mathbf{x}_n$ transformée par l'application de caractéristiques, $\phi(\mathbf{x}_n)$. En utilisant cette notation, écrivez la forme fermée du postérieur pour la régression polynomiale bayésienne en termes de $\Phi$.

2. **(Effet de la Complexité du Modèle)** Pour la régression à noyau bayésienne, vous devez pré-déterminer le nombre de caractéristiques (c'est-à-dire $D$) et l'hyperparamètre $\alpha$ dans le prior. Pour un jeu de données jouet avec 40 observations (définissez number_of_points=20), visualisez l'intervalle prédictif postérieur à 95% pour $D = [1,3,5,10,15,20,50,100]$ et $\alpha = [0.1, 1, 5, 10, 100]$ (arrangez ces visualisations dans une grille), en utilisant la régression polynomiale bayésienne.

Basé sur votre visualisation, décrivez en termes intuitifs quel est le rôle de $\alpha$ et $D$ dans la détermination de la forme de l'incertitude prédictive postérieure.
<br><br>
***Indice:*** Lisez le Problème 3 avant d'implémenter le Problème 2, vous pouvez implémenter les deux en même temps.
<br><br>
**Crédit supplémentaire:** Quand l'application de caractéristiques $\phi$ est une transformation générale (généralement non-linéaire), appliquer la régression linéaire bayésienne sur l'entrée transformée est appelé **Régression à Noyau Bayésienne**. Choisissez votre propre application de caractéristiques non-linéaire $\phi: \mathbb{R} \to \mathbb{R}^5$ et visualisez l'intervalle prédictif postérieur à 95% de la régression à noyau bayésienne pour votre choix de $\phi$ et $D = [1,3,5,10,15,20,50,100]$, $\alpha = [0.1, 1, 5, 10, 100]$. Comparez la visualisation à celle de la régression polynomiale bayésienne. Est-ce que le prédictif postérieur de votre régression à noyau bayésienne capture des propriétés importantes du prédictif postérieur du modèle de régression polynomiale bayésienne?
<br><br>
**Note:** nous recommandons fortement d'implémenter l'application de caractéristiques suivante:
<br><br>
\begin{align}
\phi: \mathbb{R}^{D'} &\to \mathbb{R}^D\\
\mathbf{x} &\mapsto \left[\sqrt{\frac{2}{D}} \cos(w_1^\top x + b_1), \ldots, \sqrt{\frac{2}{D}} \cos(w_D^\top x + b_D)\right]
\end{align}
<br>où $b_d \sim [0, 2\pi]$ et $w_d \sim N(0, \beta I_{D'\times D'})$ doivent être échantillonnés aléatoirement et fixés avant la modélisation et l'inférence. Pour cet exercice, nous suggérons de définir $\beta=10$. Les caractéristiques générées par $\phi$ sont appelées **Caractéristiques de Fourier Aléatoires** (Random Fourier Features). Quand le nombre de caractéristiques $D$ tend vers l'infini, le modèle de régression à noyau bayésienne résultant tend vers un type important de modèle bayésien (non-paramétrique) appelé modèle de Processus Gaussien. Nous revisiterons la connexion entre la régression à noyau bayésienne et les processus gaussiens dans la dernière partie du cours.

3. **(Évaluation du Modèle et Estimation de l'Incertitude)** Rappelez-vous qu'une comparaison visuelle directe de l'intervalle prédictif à 95% contre les données d'entraînement est impraticable! Plutôt, pour évaluer l'ajustement du modèle bayésien sur les données observées, nous évaluons la log-vraisemblance marginale des données sous le postérieur. Étant donné un ensemble de test $\{(\mathbf{x}^*_m, \mathbf{y}^*_m)\}$, la log-vraisemblance prédictive postérieure ou, simplement, la **log-vraisemblance** est calculée comme:
\begin{align}
\\ \log \prod_{m=1}^M p(\mathbf{y}^*_m | \mathbf{x}^*_m, \text{Données}) &= \sum_{m=1}^M \log p(\mathbf{y}^*_m | \mathbf{x}^*_m, \text{Données})\\
&= \sum_{m=1}^M \log \int_\mathbf{w} p(\mathbf{y}^*_m | \mathbf{x}^*_m, \mathbf{w}) p(\mathbf{w}| \text{Données}) d\mathbf{w}
\end{align}
<br>c'est-à-dire que la log-vraisemblance à une seule observation $(\mathbf{x}^*_m, \mathbf{y}^*_m)$ est le log de la vraisemblance de l'observation ***moyennée sur tous les modèles dans le postérieur***.
<br><br>
Pour la régression linéaire bayésienne, avec postérieur ${N}(\mu_N, \Sigma_N)$ nous avons que
$$
p(y^*_m | x^*_m, \text{Données}) = {N}(\mu^\top\mathbf{x}^*_m, \sigma^2 + (\mathbf{x}^*_m)^\top\Sigma_N\mathbf{x}^*_m)
$$
où $\sigma^2$ est la variance du bruit d'observation.
<br><br>
Pour chaque choix de $D$ et $\alpha$ dans le Problème 2, calculez la log-vraisemblance des données d'entraînement. Examinez les modèles avec les log-vraisemblances plus élevées et quelques-uns avec des log-vraisemblances plus basses, quelle est la relation entre la log-vraisemblance et l'incertitude prédictive? En particulier, est-ce qu'une log-vraisemblance plus élevée indique une "meilleure" incertitude prédictive?

2. **(Incertitude Bayésienne versus Fréquentiste)** Comparez les types d'incertitudes prédictives qui sont générées par les modèles bayésiens et les ensembles. Caractérisez les avantages et inconvénients des incertitudes bootstrap d'un ensemble. Décrivez une situation où il serait préférable de calculer les incertitudes bootstrap plutôt que les incertitudes prédictives postérieures d'un modèle bayésien.

  ***Indice:*** Par exemple, considérez des situations où les données sont rares versus des situations où les données sont abondantes; considérez des situations où les cliniciens peuvent fournir des conseils sur la sélection de modèle en utilisant l'expertise du domaine versus des situations où nous ne saurions pas comment les patterns dans les données s'extrapoleraient à de nouvelles populations de patients.

  Caractérisez les avantages et inconvénients des incertitudes prédictives postérieures d'un modèle bayésien. Décrivez une application où il est préférable d'utiliser ces incertitudes plutôt que les incertitudes bootstrap d'un ensemble.

3. **(Mesurer l'Incertitude)** D'après vos expériences, est-ce que l'une des métriques d'évaluation de modèle considérées dans ce devoir (MSE, log-vraisemblance) est appropriée pour évaluer la qualité de l'incertitude prédictive loin des données d'entraînement, c'est-à-dire, si nous sommes préoccupés par la performance des modèles sous décalage de covariables devrions-nous utiliser ces métriques pour effectuer la sélection de modèle?

  Est-ce que nos "meilleures pratiques" couramment utilisées pour entraîner des modèles d'apprentissage automatique aident ou entravent notre capacité à entraîner des modèles avec des incertitudes prédictives utiles?

  Quelle serait une bonne métrique pour mesurer l'incertitude? Comment définiriez-vous une "bonne" incertitude en premier lieu?

  ***Indice:*** Pouvez-vous formuler une définition de "bonne" incertitude sans référencer une tâche en aval spécifique?

## Partie III: Calibration et Vérifications Prédictives Postérieures
Vous pouvez utiliser ces fonctions d'aide telles quelles. Ne modifiez pas sauf si nécessaire pour le débogage.

In [ ]:
def reliability_diagram(probs, y_true, n_bins=10, title=None):
    """Tracer un diagramme de fiabilité en utilisant des bins de confiance (cas binaire)."""
    probs = np.asarray(probs)
    y_true = np.asarray(y_true)
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(probs, bins) - 1
    bin_ids = np.clip(bin_ids, 0, n_bins - 1)

    conf = np.zeros(n_bins)
    acc = np.zeros(n_bins)
    counts = np.zeros(n_bins)

    for b in range(n_bins):
        mask = bin_ids == b
        counts[b] = mask.sum()
        if counts[b] > 0:
            conf[b] = probs[mask].mean()
            acc[b] = y_true[mask].mean()
        else:
            conf[b] = np.nan
            acc[b] = np.nan

    plt.figure()
    plt.plot([0, 1], [0, 1])
    plt.scatter(conf, acc)
    plt.xlabel('Probabilité prédite moyenne')
    plt.ylabel('Fréquence empirique')
    if title is not None:
        plt.title(title)
    plt.ylim(0, 1)
    plt.xlim(0, 1)
    plt.show()

    return conf, acc, counts


def ece_binary(probs, y_true, n_bins=10):
    """Erreur de Calibration Attendue (ECE) pour la classification binaire."""
    probs = np.asarray(probs)
    y_true = np.asarray(y_true)

    bins = np.linspace(0.0, 1.0, n_bins + 1)
    bin_ids = np.digitize(probs, bins) - 1
    bin_ids = np.clip(bin_ids, 0, n_bins - 1)

    ece = 0.0
    n = len(y_true)
    for b in range(n_bins):
        mask = bin_ids == b
        if mask.sum() == 0:
            continue
        acc = y_true[mask].mean()
        conf = probs[mask].mean()
        ece += (mask.sum() / n) * abs(acc - conf)
    return float(ece)


def brier_binary(probs, y_true):
    probs = np.asarray(probs)
    y_true = np.asarray(y_true)
    return float(np.mean((probs - y_true)**2))


def temperature_scale(logits, T):
    """Mise à l'échelle de température binaire sur les logits."""
    logits = np.asarray(logits)
    return 1 / (1 + np.exp(-logits / T))

D'abord nous allons entraîner un classificateur probabiliste,
évaluer NLL, Brier, ECE, et visualiser la calibration en utilisant un diagramme de fiabilité.

**Jeu de données:** Nous générons un jeu de données de classification binaire synthétique.

In [ ]:
X, y = make_classification(
    n_samples=8000,
    n_features=20,
    n_informative=10,
    n_redundant=2,
    flip_y=0.02,
    class_sep=1.0,
    random_state=0,
)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)

clf = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(max_iter=1000))
])
clf.fit(X_train, y_train)

probs = clf.predict_proba(X_test)[:, 1]

print('NLL (log loss):', log_loss(y_test, probs))
print('Brier:', brier_binary(probs, y_test))
print('ECE:', ece_binary(probs, y_test, n_bins=15))

reliability_diagram(probs, y_test, n_bins=15, title='Diagramme de fiabilité (avant calibration)')

1. Dans le diagramme de fiabilité ci-dessus, est-ce que le modèle est **surconfiant**, **sous-confiant**, ou proche de calibré?
2. Quelle métrique parmi NLL, Brier, et ECE est *le plus directement* une métrique de calibration? Justifiez brièvement.

Écrivez votre réponse dans la cellule markdown ci-dessous.

3. Nous allons effectuer une mise à l'échelle de température sur un ensemble de calibration séparé.

**Étapes**
Divisez X_train en un ensemble d'entraînement plus petit et un ensemble de calibration. Ajustez le classificateur sur l'ensemble d'entraînement plus petit. Utilisez l'ensemble de calibration pour choisir une température T qui minimise NLL. Évaluez le diagramme de fiabilité, ECE, Brier, NLL sur l'ensemble de test avant vs après la mise à l'échelle. Qu'est-ce qui s'est amélioré et qu'est-ce qui ne s'est pas amélioré?

In [ ]:
# Diviser train en (train_small, cal)
X_tr, X_cal, y_tr, y_cal = train_test_split(X_train, y_train, test_size=0.3, random_state=1, stratify=y_train)

clf2 = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(max_iter=1000))
])
clf2.fit(X_tr, y_tr)

# Obtenir les logits pour la calibration et le test
logits_cal = clf2.named_steps['lr'].decision_function(clf2.named_steps['scaler'].transform(X_cal))
logits_test = clf2.named_steps['lr'].decision_function(clf2.named_steps['scaler'].transform(X_test))

# Recherche en grille pour T
Ts = np.logspace(-2, 2, 80)
cal_nll = []
for T in Ts:
    p = temperature_scale(logits_cal, T)
    cal_nll.append(log_loss(y_cal, p))
cal_nll = np.array(cal_nll)

best_T = Ts[np.argmin(cal_nll)]
print('Meilleur T:', best_T)

plt.figure()
plt.plot(Ts, cal_nll)
plt.xscale('log')
plt.xlabel('Température T')
plt.ylabel('NLL de calibration')
plt.title('Mise à l\'échelle de température: choisir T en minimisant NLL')
plt.show()

# Évaluer avant/après sur le test
p_before = 1 / (1 + np.exp(-logits_test))
p_after = temperature_scale(logits_test, best_T)

print('--- Métriques de test ---')
print('Avant: NLL', log_loss(y_test, p_before), 'Brier', brier_binary(p_before, y_test), 'ECE', ece_binary(p_before, y_test, n_bins=15))
print('Après : NLL', log_loss(y_test, p_after),  'Brier', brier_binary(p_after, y_test),  'ECE', ece_binary(p_after, y_test, n_bins=15))

reliability_diagram(p_before, y_test, n_bins=15, title='Diagramme de fiabilité (avant mise à l\'échelle temp)')
reliability_diagram(p_after, y_test, n_bins=15, title='Diagramme de fiabilité (après mise à l\'échelle temp)')

### **Vérifications Prédictives Postérieures (PPCs) pour la régression**

Nous ajustons un simple modèle de régression probabiliste et exécutons des vérifications de style PPC.

Nous supposons un bruit gaussien:
\[
Y \mid x \sim N(f(x), \sigma^2).
\]

Nous allons:
- ajuster un modèle de régression ridge pour la fonction moyenne, et
- estimer \(\sigma\) sur l'ensemble d'entraînement,
puis utiliser ceci comme distribution prédictive.

C'est un proxy simplifié pour pratiquer les mécaniques PPC.

In [ ]:
Xr, yr = make_regression(n_samples=4000, n_features=10, noise=15.0, random_state=0)
Xr_train, Xr_test, yr_train, yr_test = train_test_split(Xr, yr, test_size=0.3, random_state=0)

reg = Pipeline([
    ('scaler', StandardScaler()),
    ('ridge', Ridge(alpha=1.0))
])
reg.fit(Xr_train, yr_train)

mu_train = reg.predict(Xr_train)
mu_test = reg.predict(Xr_test)

# Estimer sigma à partir des résidus d'entraînement
resid = yr_train - mu_train
sigma_hat = np.std(resid)
print('Sigma estimé:', sigma_hat)

4. Implémentez la log-vraisemblance négative prédictive moyenne pour une distribution prédictive gaussienne: $ -\frac{1}{n}\sum_i \log N(y_i \mid \mu_i, \sigma^2). $ Retournez un scalaire.

In [ ]:
import math

def gaussian_nll(y, mu, sigma):
    # TODO: implémenter la log-vraisemblance négative moyenne pour N(mu, sigma^2)
    raise NotImplementedError

# Décommentez après implémentation
# print('NLL de test:', gaussian_nll(yr_test, mu_test, sigma_hat))

5. Effectuez une PPC de base: Simulez des résultats répliqués $\tilde {y}^{(s)}\sim {N}(\mu_{test}, \sigma^2)$ pour $S = 500$ répliques et comparez les statistiques $T$ entre les résultats répliqués et observés. Utilisez deux statistiques: $T_1$ = moyenne($y$); $T_2$ = variance($y$). Fournissez un graphique montrant la distribution des $T_1$ et $T_2$ répliqués avec la valeur observée marquée et une courte interprétation de si le modèle semble ajuster ces statistiques?

# Partie IV: Prédiction Conforme
Nous allons implémenter la prédiction conforme divisée (split conformal prediction) pour la régression.

Objectif: produire des intervalles de prédiction $[L(x), U(x)]$ avec une couverture marginale approximative $(1-\alpha)$.

**Prédiction conforme divisée pour la régression:**

i) Ajuster le modèle sur l'ensemble d'entraînement.

ii) Sur l'ensemble de calibration, calculer les résidus $r_i = |y_i - \hat{y}_i|$.

iii) Soit $q$ le quantile $(1-\alpha)$ de ${r_i}$.
Intervalle de sortie pour un nouveau point $x$: $ [\hat{y}(x)-q,\ \hat{y}(x)+q]. $

In [ ]:
# Préparer les divisions de données
Xr_tr, Xr_temp, yr_tr, yr_temp = train_test_split(Xr, yr, test_size=0.4, random_state=0)
Xr_cal, Xr_te, yr_cal, yr_te = train_test_split(Xr_temp, yr_temp, test_size=0.5, random_state=0)

base = Pipeline([
    ('scaler', StandardScaler()),
    ('ridge', Ridge(alpha=1.0))
])
base.fit(Xr_tr, yr_tr)

pred_cal = base.predict(Xr_cal)
pred_te = base.predict(Xr_te)

resid_cal = np.abs(yr_cal - pred_cal)

def split_conformal_interval(pred, resid_cal, alpha=0.1):
    # TODO: calculer q = quantile (1-alpha) des résidus de calibration
    # Retourner les tableaux lower, upper pour pred
    raise NotImplementedError

# Décommentez après implémentation
# L, U = split_conformal_interval(pred_te, resid_cal, alpha=0.1)
# coverage = np.mean((yr_te >= L) & (yr_te <= U))
# avg_width = np.mean(U - L)
# print('Couverture empirique:', coverage)
# print('Largeur moyenne de l\'intervalle:', avg_width)

1. Après avoir implémenté `split_conformal_interval`:

i) Calculez la couverture empirique pour $\alpha\in\{0.05, 0.1, 0.2\}$.

ii) Rapportez la largeur moyenne pour chaque $\alpha$.

iii) Interprétez brièvement le compromis.

2. En 5-8 phrases:

Sous quelles hypothèses la prédiction conforme divisée garantit-elle la couverture?

Pourquoi ces hypothèses pourraient-elles échouer sous décalage de covariables?

Quelle est l'idée générale derrière les méthodes conformes pondérées?

Écrivez votre réponse ci-dessous.